In [3]:
import os, subprocess

def load_amber_env():
    """Load AmberTools + Amber environment into the Jupyter kernel."""
    bash_cmd = """
    module purge
    module load StdEnvACCRE/2023
    module load gcc/12.3 openmpi/4.1.5 cuda/12.2
    module load ambertools/25.0
    module load scipy-stack/2025a
    module load amber/24
    env
    """
    env_output = subprocess.check_output(["bash", "-c", bash_cmd], text=True)
    for line in env_output.splitlines():
        if "=" in line:
            k, v = line.split("=", 1)
            os.environ[k] = v
    print("✅ Amber environment loaded into Jupyter kernel.")


In [4]:
load_amber_env()

bash: ml: line 1: syntax error: unexpected end of file
bash: error importing function definition for `ml'
bash: find_port: line 1: syntax error: unexpected end of file
bash: error importing function definition for `find_port'
bash: which: line 1: syntax error: unexpected end of file
bash: error importing function definition for `which'
bash: create_passwd: line 1: syntax error: unexpected end of file
bash: error importing function definition for `create_passwd'
bash: random_number: line 1: syntax error: unexpected end of file
bash: error importing function definition for `random_number'
bash: module: line 1: syntax error: unexpected end of file
bash: error importing function definition for `module'
bash: setup_accre_software_stack: line 1: syntax error: unexpected end of file
bash: error importing function definition for `setup_accre_software_stack'
bash: wait_until_port_used: line 1: syntax error: unexpected end of file
bash: error importing function definition for `wait_until_port_us

✅ Amber environment loaded into Jupyter kernel.


## 🧬 **Cα Publish Pipeline — Summary**

For each system, the workflow:

- 🔍 Reads the three `combined*_nowat_ions.nc` trajectories with the matching stripped topology.  
- ✂️ Strips all atoms except `:1–275@CA` and writes a correct **Cα-only topology**.  
- 🔗 Generates three Cα-only replicate trajectories, concatenates them into **`Ca_combined.nc`**, and deletes the intermediate files.  
- 🔒 Leaves the original data untouched and writes only **`Ca_combined.prmtop`** and **`Ca_combined.nc`** into the publish directory.


In [6]:
%%bash
set -euo pipefail

SRC_BASE="/panfs/accrepfs.vampire/data/kojetin_lab/aMD_Analysis"
OUT_BASE="/panfs/accrepfs.vampire/data/kojetin_lab/Mithun/Publish_aMD_12_27_2025"
mkdir -p "$OUT_BASE"

systems=(
  3PRG
  6ONI_apo
  6ONI_pep_nolig
  6ONI_ligand
  6ONI_pep_lig
  6ONJ_apo
  6ONJ_pep_nolig
  6ONJ_ligand
  6ONJ_pep_lig
)

echo "🧬 Publishing CA-only trajectories into:"
echo "    $OUT_BASE"
echo

for sys in "${systems[@]}"; do
  SRC="${SRC_BASE}/${sys}"
  OUT="${OUT_BASE}/${sys}"
  mkdir -p "$OUT"

  echo "➡️  ${sys}"

  PRMTOP="${SRC}/${sys}_rep1_nowater.prmtop"
  TRAJ1="${SRC}/${sys}_rep1_nowater.nc"
  TRAJ2="${SRC}/${sys}_rep2_nowater.nc"
  TRAJ3="${SRC}/${sys}_rep3_nowater.nc"

  # -----------------------------------------
  # Stage 1: generate CA-only reps + CA.prmtop
  # -----------------------------------------
  cpptraj <<EOF
parm ${PRMTOP}

trajin ${TRAJ1}
autoimage
strip !(:1-275@CA) parmout ${OUT}/Ca.prmtop
trajout ${OUT}/Ca_rep1.nc netcdf
run
clear trajin
clear trajout

trajin ${TRAJ2}
autoimage
strip !(:1-275@CA)
trajout ${OUT}/Ca_rep2.nc netcdf
run
clear trajin
clear trajout

trajin ${TRAJ3}
autoimage
strip !(:1-275@CA)
trajout ${OUT}/Ca_rep3.nc netcdf
run
quit
EOF

  # -----------------------------------------
  # Stage 2: concatenate using the CA.prmtop
  # -----------------------------------------
  cpptraj <<EOF
parm ${OUT}/Ca.prmtop
trajin ${OUT}/Ca_rep1.nc
trajin ${OUT}/Ca_rep2.nc
trajin ${OUT}/Ca_rep3.nc
trajout ${OUT}/Ca_combined.nc netcdf
run
quit
EOF

  rm -f ${OUT}/Ca_rep1.nc ${OUT}/Ca_rep2.nc ${OUT}/Ca_rep3.nc
  mv -f ${OUT}/Ca.prmtop ${OUT}/Ca_combined.prmtop

  echo "   ✅ ${OUT}/Ca_combined.prmtop"
  echo "   ✅ ${OUT}/Ca_combined.nc"
  echo
done

echo "🎯 Done: all systems CA-stripped + concatenated using *_rep*_nowater.* files."


bash: ml: line 1: syntax error: unexpected end of file
bash: error importing function definition for `ml'
bash: find_port: line 1: syntax error: unexpected end of file
bash: error importing function definition for `find_port'
bash: which: line 1: syntax error: unexpected end of file
bash: error importing function definition for `which'
bash: create_passwd: line 1: syntax error: unexpected end of file
bash: error importing function definition for `create_passwd'
bash: random_number: line 1: syntax error: unexpected end of file
bash: error importing function definition for `random_number'
bash: module: line 1: syntax error: unexpected end of file
bash: error importing function definition for `module'
bash: setup_accre_software_stack: line 1: syntax error: unexpected end of file
bash: error importing function definition for `setup_accre_software_stack'
bash: wait_until_port_used: line 1: syntax error: unexpected end of file
bash: error importing function definition for `wait_until_port_us

🧬 Publishing CA-only trajectories into:
    /panfs/accrepfs.vampire/data/kojetin_lab/Mithun/Publish_aMD_12_27_2025

➡️  3PRG

CPPTRAJ: Trajectory Analysis. V6.29.13 (AmberTools) OpenMP
    ___  ___  ___  ___
     | \/ | \/ | \/ | 
    _|_/\_|_/\_|_/\_|_

| 16 OpenMP threads available.
| Date/time: 12/29/25 01:24:52
| Available memory: 5.386 GB

INPUT: Reading input from 'STDIN'
  [parm /panfs/accrepfs.vampire/data/kojetin_lab/aMD_Analysis/3PRG/3PRG_rep1_nowater.prmtop]
	Reading '/panfs/accrepfs.vampire/data/kojetin_lab/aMD_Analysis/3PRG/3PRG_rep1_nowater.prmtop' as Amber Topology
	Radius Set: modified Bondi radii (mbondi)
  [trajin /panfs/accrepfs.vampire/data/kojetin_lab/aMD_Analysis/3PRG/3PRG_rep1_nowater.nc]
	Reading '/panfs/accrepfs.vampire/data/kojetin_lab/aMD_Analysis/3PRG/3PRG_rep1_nowater.nc' as Amber NetCDF
  [autoimage]
    AUTOIMAGE: To box center based on center of mass, anchor is first molecule.
	Auto-imaging using molecule distances.
  [strip !(:1-275@CA) parmout /panfs/a

## 🧬 **Master Cα Concatenation — Round 2**

This step generates only two master trajectories in  
`Publish_aMD_12_27_2025/`:

- **6ONI_ALL_Ca_combined.nc** — concatenation of all 6ONI systems  
- **6ONJ_ALL_Ca_combined.nc** — concatenation of all 6ONJ systems  

The **apo `Ca_combined.prmtop`** is used as the reference topology for both.

⚠️ **3PRG is not concatenated** because it exists only as a single apo system — its `Ca_combined.nc` inside the `3PRG/` folder is already final.


In [8]:
%%bash
set -euo pipefail

BASE="/panfs/accrepfs.vampire/data/kojetin_lab/Mithun/Publish_aMD_12_27_2025"
OUTBASE="${BASE}/6ONI_6ONJ_ALL_PCA"

mkdir -p ${OUTBASE}/6ONI
mkdir -p ${OUTBASE}/6ONJ

echo "🧬 Creating master Cα trajectories in:"
echo "    $OUTBASE"
echo

# -------------------------
# 6ONI — concatenate all
# -------------------------
cpptraj <<EOF
parm ${BASE}/6ONI_apo/Ca_combined.prmtop
trajin ${BASE}/6ONI_apo/Ca_combined.nc
trajin ${BASE}/6ONI_pep_nolig/Ca_combined.nc
trajin ${BASE}/6ONI_ligand/Ca_combined.nc
trajin ${BASE}/6ONI_pep_lig/Ca_combined.nc
trajout ${OUTBASE}/6ONI/6ONI_ALL_Ca_combined.nc netcdf
run
quit
EOF
echo "✅ 6ONI_ALL_Ca_combined.nc created"

# -------------------------
# 6ONJ — concatenate all
# -------------------------
cpptraj <<EOF
parm ${BASE}/6ONJ_apo/Ca_combined.prmtop
trajin ${BASE}/6ONJ_apo/Ca_combined.nc
trajin ${BASE}/6ONJ_pep_nolig/Ca_combined.nc
trajin ${BASE}/6ONJ_ligand/Ca_combined.nc
trajin ${BASE}/6ONJ_pep_lig/Ca_combined.nc
trajout ${OUTBASE}/6ONJ/6ONJ_ALL_Ca_combined.nc netcdf
run
quit
EOF
echo "✅ 6ONJ_ALL_Ca_combined.nc created"

echo
echo "🎯 Done — master CA trajectories written."


bash: ml: line 1: syntax error: unexpected end of file
bash: error importing function definition for `ml'
bash: find_port: line 1: syntax error: unexpected end of file
bash: error importing function definition for `find_port'
bash: which: line 1: syntax error: unexpected end of file
bash: error importing function definition for `which'
bash: create_passwd: line 1: syntax error: unexpected end of file
bash: error importing function definition for `create_passwd'
bash: random_number: line 1: syntax error: unexpected end of file
bash: error importing function definition for `random_number'
bash: module: line 1: syntax error: unexpected end of file
bash: error importing function definition for `module'
bash: setup_accre_software_stack: line 1: syntax error: unexpected end of file
bash: error importing function definition for `setup_accre_software_stack'
bash: wait_until_port_used: line 1: syntax error: unexpected end of file
bash: error importing function definition for `wait_until_port_us

🧬 Creating master Cα trajectories in:
    /panfs/accrepfs.vampire/data/kojetin_lab/Mithun/Publish_aMD_12_27_2025/6ONI_6ONJ_ALL_PCA


CPPTRAJ: Trajectory Analysis. V6.29.13 (AmberTools) OpenMP
    ___  ___  ___  ___
     | \/ | \/ | \/ | 
    _|_/\_|_/\_|_/\_|_

| 16 OpenMP threads available.
| Date/time: 12/29/25 01:58:35
| Available memory: 6.393 GB

INPUT: Reading input from 'STDIN'
  [parm /panfs/accrepfs.vampire/data/kojetin_lab/Mithun/Publish_aMD_12_27_2025/6ONI_apo/Ca_combined.prmtop]
	Reading '/panfs/accrepfs.vampire/data/kojetin_lab/Mithun/Publish_aMD_12_27_2025/6ONI_apo/Ca_combined.prmtop' as Amber Topology
	Radius Set: modified Bondi radii (mbondi)
  [trajin /panfs/accrepfs.vampire/data/kojetin_lab/Mithun/Publish_aMD_12_27_2025/6ONI_apo/Ca_combined.nc]
	Reading '/panfs/accrepfs.vampire/data/kojetin_lab/Mithun/Publish_aMD_12_27_2025/6ONI_apo/Ca_combined.nc' as Amber NetCDF
  [trajin /panfs/accrepfs.vampire/data/kojetin_lab/Mithun/Publish_aMD_12_27_2025/6ONI_pep_nolig/Ca_combin

### Create global Cα master trajectory

Combine the Cα trajectories in the order **6ONI_ALL → 6ONJ_ALL → 3PRG** into a single NetCDF trajectory: `ALL_SYSTEMS_Ca_combined.nc`.

In [9]:
%%bash
set -euo pipefail

BASE="/panfs/accrepfs.vampire/data/kojetin_lab/Mithun/Publish_aMD_12_27_2025"
OUTDIR="${BASE}/6ONI_6ONJ_ALL_PCA/ALL"

mkdir -p "$OUTDIR"

echo "🧬 Creating global Cα master trajectory:"
echo "    Order: 6ONI_ALL → 6ONJ_ALL → 3PRG"
echo

cpptraj <<EOF
parm ${BASE}/6ONI_apo/Ca_combined.prmtop
trajin ${BASE}/6ONI_6ONJ_ALL_PCA/6ONI/6ONI_ALL_Ca_combined.nc
trajin ${BASE}/6ONI_6ONJ_ALL_PCA/6ONJ/6ONJ_ALL_Ca_combined.nc
trajin ${BASE}/3PRG/Ca_combined.nc
trajout ${OUTDIR}/ALL_SYSTEMS_Ca_combined.nc netcdf
run
quit
EOF

echo "🎯 ${OUTDIR}/ALL_SYSTEMS_Ca_combined.nc created successfully."


bash: ml: line 1: syntax error: unexpected end of file
bash: error importing function definition for `ml'
bash: find_port: line 1: syntax error: unexpected end of file
bash: error importing function definition for `find_port'
bash: which: line 1: syntax error: unexpected end of file
bash: error importing function definition for `which'
bash: create_passwd: line 1: syntax error: unexpected end of file
bash: error importing function definition for `create_passwd'
bash: random_number: line 1: syntax error: unexpected end of file
bash: error importing function definition for `random_number'
bash: module: line 1: syntax error: unexpected end of file
bash: error importing function definition for `module'
bash: setup_accre_software_stack: line 1: syntax error: unexpected end of file
bash: error importing function definition for `setup_accre_software_stack'
bash: wait_until_port_used: line 1: syntax error: unexpected end of file
bash: error importing function definition for `wait_until_port_us

🧬 Creating global Cα master trajectory:
    Order: 6ONI_ALL → 6ONJ_ALL → 3PRG


CPPTRAJ: Trajectory Analysis. V6.29.13 (AmberTools) OpenMP
    ___  ___  ___  ___
     | \/ | \/ | \/ | 
    _|_/\_|_/\_|_/\_|_

| 16 OpenMP threads available.
| Date/time: 12/29/25 02:00:34
| Available memory: 8.243 GB

INPUT: Reading input from 'STDIN'
  [parm /panfs/accrepfs.vampire/data/kojetin_lab/Mithun/Publish_aMD_12_27_2025/6ONI_apo/Ca_combined.prmtop]
	Reading '/panfs/accrepfs.vampire/data/kojetin_lab/Mithun/Publish_aMD_12_27_2025/6ONI_apo/Ca_combined.prmtop' as Amber Topology
	Radius Set: modified Bondi radii (mbondi)
  [trajin /panfs/accrepfs.vampire/data/kojetin_lab/Mithun/Publish_aMD_12_27_2025/6ONI_6ONJ_ALL_PCA/6ONI/6ONI_ALL_Ca_combined.nc]
	Reading '/panfs/accrepfs.vampire/data/kojetin_lab/Mithun/Publish_aMD_12_27_2025/6ONI_6ONJ_ALL_PCA/6ONI/6ONI_ALL_Ca_combined.nc' as Amber NetCDF
  [trajin /panfs/accrepfs.vampire/data/kojetin_lab/Mithun/Publish_aMD_12_27_2025/6ONI_6ONJ_ALL_PCA/6ONJ/6ONJ_ALL

# 🧠 Job 1 — PCA for each entire system including 6ONI all, 6ONJ all and ALL 

In [11]:
%%bash
set -euo pipefail

BASE="/panfs/accrepfs.vampire/data/kojetin_lab/Mithun/Publish_aMD_12_27_2025"
SLURM_DIR="${BASE}/Slurm_Scripts"
mkdir -p "$SLURM_DIR"
cd "$SLURM_DIR"

cat > PCA_ALL_GROUPS.slurm <<'EOF'
#!/bin/bash
#SBATCH --job-name=PCA_ALL_GROUPS
#SBATCH --mail-user=mithun.nag.karadi.giridhar@vanderbilt.edu
#SBATCH --mail-type=ALL
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=1
#SBATCH --mem=60G
#SBATCH --gres=gpu:nvidia_titan_x:1
#SBATCH --time=5-00:00:00
#SBATCH --partition=batch_gpu
#SBATCH --account=csb_gpu_acc
#SBATCH --output=PCA_ALL_GROUPS_out_%j.log
#SBATCH --error=PCA_ALL_GROUPS_err_%j.log

set -euo pipefail

module --force purge
module load StdEnvACCRE/2023
module load ambertools/25.0
module load amber/24
export OMP_NUM_THREADS=1

BASE="/panfs/accrepfs.vampire/data/kojetin_lab/Mithun/Publish_aMD_12_27_2025"

# =====================================================
# GROUP 3 — GLOBAL PCA (aMD master trajectory)
# =====================================================
echo "===== PCA GROUP 3 : GLOBAL aMD PCA ====="

cd "${BASE}/6ONI_6ONJ_ALL_PCA/ALL"

cpptraj <<EOT
parm ${BASE}/6ONI_apo/Ca_combined.prmtop
trajin ${BASE}/6ONI_6ONJ_ALL_PCA/ALL/ALL_SYSTEMS_Ca_combined.nc
autoimage anchor :1-275@CA
rms first :1-275@CA
average :1-275@CA crdset avgstruct
createcrd trajectory
run
crdaction trajectory rms ref avgstruct :1-275@CA
crdaction trajectory matrix covar name covmatrix :1-275@CA
runanalysis diagmatrix covmatrix out evecs.dat vecs 50 name AllModes
crdaction trajectory projection modes AllModes beg 1 end 50 :1-275@CA out pca.dat
quit
EOT

# =====================================================
# GROUP 2 — 6ONI_ALL and 6ONJ_ALL
# =====================================================
echo "===== PCA GROUP 2 : 6ONI_ALL and 6ONJ_ALL ====="

for SYS in 6ONI 6ONJ; do
  PCADIR="${BASE}/6ONI_6ONJ_ALL_PCA/${SYS}_ALL_PCA"
  mkdir -p "$PCADIR"
  cd "$PCADIR"

  cpptraj <<EOT
parm ${BASE}/${SYS}_apo/Ca_combined.prmtop
trajin ${BASE}/6ONI_6ONJ_ALL_PCA/${SYS}/${SYS}_ALL_Ca_combined.nc
autoimage anchor :1-275@CA
rms first :1-275@CA
average :1-275@CA crdset avgstruct
createcrd trajectory
run
crdaction trajectory rms ref avgstruct :1-275@CA
crdaction trajectory matrix covar name covmatrix :1-275@CA
runanalysis diagmatrix covmatrix out evecs.dat vecs 50 name Modes
crdaction trajectory projection modes Modes beg 1 end 50 :1-275@CA out pca.dat
quit
EOT
done

# =====================================================
# GROUP 1 — INDIVIDUAL SYSTEMS
# =====================================================
echo "===== PCA GROUP 1 : INDIVIDUAL SYSTEMS ====="

systems=(3PRG 6ONI_apo 6ONI_pep_nolig 6ONI_ligand 6ONI_pep_lig 6ONJ_apo 6ONJ_pep_nolig 6ONJ_ligand 6ONJ_pep_lig)

for sys in "${systems[@]}"; do
  PCADIR="${BASE}/${sys}/PCA"
  mkdir -p "$PCADIR"
  cd "$PCADIR"

  cpptraj <<EOT
parm ../Ca_combined.prmtop
trajin ../Ca_combined.nc
autoimage anchor :1-275@CA
rms first :1-275@CA
average :1-275@CA crdset avgstruct
createcrd trajectory
run
crdaction trajectory rms ref avgstruct :1-275@CA
crdaction trajectory matrix covar name covmatrix :1-275@CA
runanalysis diagmatrix covmatrix out evecs.dat vecs 50 name Modes
crdaction trajectory projection modes Modes beg 1 end 50 :1-275@CA out pca.dat
quit
EOT
done

echo "🎯 ALL PCA GROUPS COMPLETED SUCCESSFULLY"
EOF

sbatch PCA_ALL_GROUPS.slurm


bash: ml: line 1: syntax error: unexpected end of file
bash: error importing function definition for `ml'
bash: find_port: line 1: syntax error: unexpected end of file
bash: error importing function definition for `find_port'
bash: which: line 1: syntax error: unexpected end of file
bash: error importing function definition for `which'
bash: create_passwd: line 1: syntax error: unexpected end of file
bash: error importing function definition for `create_passwd'
bash: random_number: line 1: syntax error: unexpected end of file
bash: error importing function definition for `random_number'
bash: module: line 1: syntax error: unexpected end of file
bash: error importing function definition for `module'
bash: setup_accre_software_stack: line 1: syntax error: unexpected end of file
bash: error importing function definition for `setup_accre_software_stack'
bash: wait_until_port_used: line 1: syntax error: unexpected end of file
bash: error importing function definition for `wait_until_port_us

Submitted batch job 8014883
